In [1]:
import os
from datetime import datetime
import pickle
import random
import math
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, MaxAbsScaler, MinMaxScaler

import torch
import torch.nn as nn
import torch.nn.functional as F



In [2]:
df = pd.read_csv(f"BTCUSDT-15m-data.csv")
print(df)

                  timestamp      open_time       open       high        low  \
0       2017-08-17 04:00:00  1502942400000    4261.48    4280.56    4261.48   
1       2017-08-17 04:15:00  1502943300000    4261.48    4270.41    4261.32   
2       2017-08-17 04:30:00  1502944200000    4280.00    4310.07    4267.99   
3       2017-08-17 04:45:00  1502945100000    4310.07    4313.62    4291.37   
4       2017-08-17 05:00:00  1502946000000    4308.83    4328.69    4304.31   
...                     ...            ...        ...        ...        ...   
275995  2025-07-07 00:00:00  1751846400000  109203.85  109273.07  109103.19   
275996  2025-07-07 00:15:00  1751847300000  109273.07  109288.02  108972.69   
275997  2025-07-07 00:30:00  1751848200000  109004.81  109037.82  108829.17   
275998  2025-07-07 00:45:00  1751849100000  108847.17  108922.00  108800.01   
275999  2025-07-07 01:00:00  1751850000000  108823.07  108823.08  108679.75   

            close     volume     close_time      qu

**Check missing values**

In [3]:
print(df.isnull())
print(f"Counts how many missing values there are in each column: {df.isnull().sum()}")
print(f"Total missing values: {df.isnull().sum().sum()}")

        timestamp  open_time   open   high    low  close  volume  close_time  \
0           False      False  False  False  False  False   False       False   
1           False      False  False  False  False  False   False       False   
2           False      False  False  False  False  False   False       False   
3           False      False  False  False  False  False   False       False   
4           False      False  False  False  False  False   False       False   
...           ...        ...    ...    ...    ...    ...     ...         ...   
275995      False      False  False  False  False  False   False       False   
275996      False      False  False  False  False  False   False       False   
275997      False      False  False  False  False  False   False       False   
275998      False      False  False  False  False  False   False       False   
275999      False      False  False  False  False  False   False       False   

        quote_av  trades  tb_base_av  t

**Split data**

In [4]:
train_end_idx = 240_000

df_train = df.iloc[:train_end_idx].copy()
df_test = df.iloc[train_end_idx:].copy()

df_test.reset_index(drop=True, inplace=True)

print(f"Length of df_train: {len(df_train)}")
print(f"Length of df_test: {len(df_test)}")

Length of df_train: 240000
Length of df_test: 36000


**Building Meaningful Features**

In [5]:
def build_features(opens, closes):
    feature1 = (closes - opens) / opens

    # Stack features
    features = np.stack([
        feature1,
        # ...
    ], axis=-1) # shape: (time_steps, num_features)

    num_features = features.shape[-1]
    return features, num_features

**Preprocess data**

In [6]:
def preprocess_data(seq_len, df):
    m = len(df)

    opens = np.array(df['open'].values)
    closes = np.array(df['close'].values)

    # Build features
    features, num_features = build_features(opens, closes)

    # Calculate number of samples
    num_samples = m - seq_len

    # Create storage for inputs & targets
    X = np.zeros([num_samples, seq_len, num_features], dtype=np.float32)
    Y = np.zeros([num_samples, num_features], dtype=np.float32)

    # Create samples (X, Y)
    for i in range(num_samples):
        X[i] = features[i : i+seq_len]
        Y[i] = features[i+seq_len : i+seq_len+1]

    return X, Y, num_features

In [7]:
# Sequence Length
seq_len = 96

# Preprocess data
X_train, Y_train, num_features = preprocess_data(seq_len, df_train)
X_test, Y_test, num_features = preprocess_data(seq_len, df_test)


In [8]:
m_train = X_train.shape[0]
m_test = X_test.shape[0]
print(f"m_train: {m_train}")
print(f"m_test: {m_test}")
print(f"X_train shape: {X_train.shape}")
print(f"Y_train shape: {Y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Y_test shape: {Y_test.shape}")

m_train: 239904
m_test: 35904
X_train shape: (239904, 96, 1)
Y_train shape: (239904, 1)
X_test shape: (35904, 96, 1)
Y_test shape: (35904, 1)


**Transform data to Torch Tensor**

In [9]:
device = torch.device("cuda:0" if torch.cuda.is_available() else 'cpu')
print("my deive: ", device)

X_train = torch.from_numpy(X_train.astype(np.float32)).to(device, dtype=torch.float32)
Y_train = torch.from_numpy(Y_train.astype(np.float32)).to(device, dtype=torch.float32)

X_test = torch.from_numpy(X_test.astype(np.float32)).to(device, dtype=torch.float32)
Y_test = torch.from_numpy(Y_test.astype(np.float32)).to(device, dtype=torch.float32)

Y_pred_test = torch.zeros([m_test, num_features], device=device, dtype=torch.float32)

my deive:  cuda:0
